In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile
import pandas as pd
import os


my_folder = '/content/drive/MyDrive/TransformesCod/02/ProductRecommendations'
file_path = '/content/drive/MyDrive/TransformesCod/02/ProductRecommendations/flipkart_com-ecommerce_sample[1].zip'

with zipfile.ZipFile(file_path, 'r') as zip_ref:
    zip_ref.extractall(my_folder)

data_csv = os.path.join(my_folder, 'flipkart_com-ecommerce_sample.csv')
df = pd.read_csv(data_csv)


In [7]:
df = df.astype(str)

df.map(lambda x: x.lower())
df.head(1)

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."


In [ ]:
df['product_specifications'][0]

'{"product_specification"=>[{"key"=>"Number of Contents in Sales Package", "value"=>"Pack of 3"}, {"key"=>"Fabric", "value"=>"Cotton Lycra"}, {"key"=>"Type", "value"=>"Cycling Shorts"}, {"key"=>"Pattern", "value"=>"Solid"}, {"key"=>"Ideal For", "value"=>"Women\'s"}, {"value"=>"Gentle Machine Wash in Lukewarm Water, Do Not Bleach"}, {"key"=>"Style Code", "value"=>"ALTHT_3P_21"}, {"value"=>"3 shorts"}]}'

In [9]:
import re

def ex_specifications(specifications):
  pairs = re.findall(r'"key"=>"(.*?)", "value"=>"(.*?)"' , specifications)
  pairs_formated = [f'{key}:{value}' for key , value in pairs]
  return ' '.join(pairs_formated)

In [10]:
df['product_specifications'] = df['product_specifications'].apply(ex_specifications)

In [11]:
df['product_specifications'][0]

"Number of Contents in Sales Package:Pack of 3 Fabric:Cotton Lycra Type:Cycling Shorts Pattern:Solid Ideal For:Women's Style Code:ALTHT_3P_21"

In [12]:
df = df.applymap(lambda x: re.sub('[^A-Za-z0-9]+' , ' ' , str(x)))
df.head(1)

/tmp/ipykernel_1889/4001588275.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub('[^A-Za-z0-9]+' , ' ' , str(x)))


,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016 03 25 22 59 23 0000,http www flipkart com alisha solid women s cyc...,Alisha Solid Women s Cycling Shorts,Clothing Women s Clothing Lingerie Sleep Swim...,SRTEH2FF9KEDEFGF,999 0,379 0,http img5a flixcart com image short u 4 a alt...,False,Key Features of Alisha Solid Women s Cycling S...,No rating available,No rating available,Alisha,Number of Contents in Sales Package Pack of 3 ...


In [ ]:
df['combined_text'] = df['product_name'] + ' '  + df['description'] + ' ' + df['product_specifications']

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sentence_embeddings = model.encode(df['combined_text'].tolist())
sentence_embeddings

array([[-0.00304955,  0.03690217,  0.01367578, ...,  0.05384491,
         0.00339341,  0.00514963],
       [ 0.01610271, -0.0764845 , -0.00826636, ...,  0.02432662,
         0.00441218, -0.04029054],
       [-0.01772362, -0.07143312, -0.00106924, ...,  0.06095115,
        -0.04368509, -0.01308934],
       ...,
       [ 0.03992024, -0.08731876, -0.00806233, ...,  0.00322414,
        -0.05832373, -0.04899615],
       [ 0.03741642, -0.08389911, -0.00784432, ..., -0.00015672,
        -0.06673806, -0.05485628],
       [ 0.04048257, -0.07920908, -0.00262117, ...,  0.00300625,
        -0.06836273, -0.05466717]], dtype=float32)

In [ ]:
import numpy as np
emb_path = my_folder + 'embeddings.npy'
np.save(emb_path , sentence_embeddings)

In [ ]:
my_folder1 = '/content/drive/MyDrive/TransformesCod/02/ProductRecommendations/'

import numpy as np
emb_path = my_folder1 + 'embeddings11.npy'
np.save(emb_path , sentence_embeddings)